### Running baseline models on the Preprocessed Data

In [78]:
import torch
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import RandomForestRegressor

from sklearn.metrics import mean_squared_error, r2_score

import os
#os.chdir('C:/GU/Thesis/code')

INPUT_DIR = 'preprocessed/eeg/'
SEED = 1234

In [79]:
cases_master = pd.read_csv('cases_data.csv')

ids = cases_master['caseid'].tolist()

In [80]:
# Extract and concatenate case files
dfs = []
ys = []
for id in ids:
    sample_path = os.path.join(INPUT_DIR, f'case_{id}.pt')
    sample_tensor = torch.load(sample_path)
    sample_df = pd.DataFrame(sample_tensor['features'].numpy())
    y = sample_tensor['bis'].numpy()

    sample_df['caseid'] = id
    dfs.append(sample_df)
    ys.append(y)

cases_df = pd.concat(dfs, ignore_index=True)
cases_df.rename(columns={
    0: '0', 1: '1', 2: '2', 3: '3', 4: '4', 5: '5', 6: '6', 7: '7', 8: '8', 9: '9',
    10: '10', 11: '11', 12: '12', 13: '13', 14: '14', 15: 'MBP', 16: 'HR', 17: 'SpO2',
    18: 'EtCO2', 19: 'BIS'
}, inplace=True)

y = np.concatenate(ys)

In [81]:
#Find all NaN values
nan_idx = (np.isnan(y))

y = y[~nan_idx]
cases_df = cases_df.iloc[np.where(~nan_idx)[0], :]

print(len(y))

print(cases_df.shape)
print(cases_df.columns)

1891721
(1891721, 20)
Index(['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
       '13', '14', 'MBP', 'HR', 'SpO2', 'EtCO2', 'caseid'],
      dtype='str')


In [82]:
# Concatenate patient context to each sequential second
cases_context = pd.merge(cases_master, cases_df, on='caseid', how='inner')

print(cases_context.columns)

X = cases_context.drop(columns=['caseid'])

Index(['caseid', 'optype', 'sex', 'age', 'asa', 'bmi', 'preop_hb', 'preop_k',
       'preop_na', 'preop_gluc', 'preop_cr', 'preop_alb', 'propofol',
       'volatile', 'remifentanil', '0', '1', '2', '3', '4', '5', '6', '7', '8',
       '9', '10', '11', '12', '13', '14', 'MBP', 'HR', 'SpO2', 'EtCO2'],
      dtype='str')


In [ ]:
#Split into train/test data
binary_cols = ['propofol', 'volatile', 'remifentanil']
X[binary_cols] = X[binary_cols].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.80, random_state=SEED
)

In [ ]:
categorical_cols = ['optype', 'sex', 'asa']
numerical_cols = ['age', 'bmi', 'preop_hb', 'preop_k', 'preop_na', 'preop_gluc', 'preop_cr', 'preop_alb']

preprocessor = ColumnTransformer([
    ('stdsca', StandardScaler(), numerical_cols),
    ('catenc', OneHotEncoder(handle_unknown='ignore'), categorical_cols)
])

rf = RandomForestRegressor(n_estimators=100, max_depth=8, n_jobs=-1)

pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('rf', rf)
])

In [ ]:
pipe.fit(X_train, y_train)
y_pred = pipe.predict(X_test)

y_pred_train = pipe.predict(X_train)

In [88]:
print(mean_squared_error(y_train, y_pred_train))
print(mean_squared_error(y_test, y_pred))

219.56015176384133
218.71594661573442
